# Notebook: Decision Support System for Credit Approval
Romain Sebire - 125 009 460

**Objective:** Build a classification model to predict whether a credit applicant will default, as part of a Kaggle in-class competition.

**Dataset:** 20,000 training samples and 5,000 test samples with 42 features describing applicant demographics, employment, and financial history.

**Approach:** Feature importance analysis with LightGBM, custom preprocessing pipeline, and hyperparameter optimization.

**Key Result:** Best accuracy of **0.5996** on the Kaggle leaderboard using LightGBM with optimized hyperparameters.

## Table of Contents

1. [Library and Data Import](#Library-and-Data-Import)
2. [Exploratory Data Analysis](#Exploratory-Data-Analysis)
3. [Feature Importance with LGBMClassifier](#Feature-Importance-with-LGBMClassifier)
4. [Data Preprocessing and Scaling](#Data-Preprocessing,-Missing-Value-Treatment-(Imputers),-and-Scaling)
5. [Model Training and Cross-Validation](#Model-Training-and-Accuracy-/-Cross-Validation-Computation)
6. [Optimized Hyperparameter Search](#Optimized-Hyperparameter-Search)
7. [Prediction and Submission](#Prediction-on-the-Test-Set-and-Submission-File-Preparation)
8. [Conclusion](#Conclusion)

## Feature Glossary

Since this dataset originates from a Brazilian Kaggle competition, all feature names are in Portuguese. Below is the translation of the most important features:

| Portuguese | English | Type |
|------------|---------|------|
| `inadimplente` | **defaulter** (target) | binary |
| `idade` | age | numerical |
| `sexo` | gender | categorical |
| `estado_civil` | marital status | categorical |
| `qtde_dependentes` | number of dependents | numerical |
| `grau_instrucao` | education level | categorical |
| `renda_mensal` | monthly income | numerical |
| `meses_na_residencia` | months at current residence | numerical |
| `meses_no_trabalho` | months at current job | numerical |
| `profissao` | profession | categorical |
| `profissao_companheiro` | partner's profession | categorical |
| `grau_instrucao_companheiro` | partner's education level | categorical |
| `tipo_residencia` | residence type | categorical |
| `nacionalidade` | nationality | categorical |
| `possui_telefone_celular` | has mobile phone | binary |
| `qtde_contas_bancarias` | number of bank accounts | numerical |
| `qtde_contas_bancarias_especiais` | number of special bank accounts | numerical |
| `estado_onde_trabalha` | state of employment | categorical |
| `estado_onde_nasceu` | state of birth | categorical |
| `forma_envio_solicitacao` | application submission method | categorical |

## Library and Data Import

In this cell, we import the essential libraries for data analysis, visualization, preprocessing, training, and evaluation of machine learning models.

In [ ]:
# --- Initial imports ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer

In [ ]:
# --- Load data ---
train_df = pd.read_csv('data/conjunto_de_treinamento.csv')
test_df = pd.read_csv('data/conjunto_de_teste.csv')
sample_submission = pd.read_csv('data/exemplo_arquivo_respostas.csv')

print("Dimensões do conjunto de treinamento:", train_df.shape)
print("Dimensões do conjunto de teste:", test_df.shape)

The data was imported correctly: there are 20,000 rows in the training set and 5,000 rows in the test set.
There are 42 columns in the training set and only 41 in the test set, as the target column is absent.

## Exploratory Data Analysis

### Feature Display and Missing Value Count

In [ ]:
# Display all columns
pd.set_option('display.max_columns', None)

# Exibição das primeiras linhas
df = train_df.copy()
df.head()

In [ ]:
# Display feature list and types
df.info()

In [ ]:
# Display target variable distribution
df['inadimplente'].value_counts(normalize=True)

The target variable is perfectly balanced: 50% are good payers and 50% are defaulters.

In [ ]:
# Display features with missing values (train_df)
missing = train_df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

In [ ]:
# Display features with missing values (test_df)
missing = test_df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

Only 6 out of 42 features have missing values; profissao_companheiro and grau_instrucao_companheiro have more than half of their rows missing. These missing values will need to be filled with an appropriate method.

### Distribution Plots for Each Feature

Let's analyze the distribution of each feature. This allows us to visually explore the data distribution within each feature, which is useful for understanding its structure, detecting imbalances or anomalies, and guiding data treatment decisions.

In [ ]:
# List of non-continuous features for barplot display
features_nao_contínuas = [
    'produto_solicitado', 
    'dia_vencimento', 
    'forma_envio_solicitacao',
    'tipo_endereco', 
    'sexo', 
    'estado_civil', 
    'grau_instrucao', 
    'nacionalidade',
    'estado_onde_nasceu', 
    'estado_onde_reside', 
    'possui_telefone_residencial',
    'codigo_area_telefone_residencial', 
    'tipo_residencia', 
    'possui_telefone_celular',
    'possui_email', 
    'possui_cartao_visa', 
    'possui_cartao_mastercard',
    'possui_cartao_diners', 
    'possui_cartao_amex', 
    'possui_outros_cartoes',
    'qtde_contas_bancarias', 
    'qtde_contas_bancarias_especiais', 
    'possui_carro',
    'vinculo_formal_com_empresa', 
    'estado_onde_trabalha', 
    'possui_telefone_trabalho',
    'codigo_area_telefone_trabalho', 
    'meses_no_trabalho', 'profissao',
    'ocupacao', 
    'profissao_companheiro', 
    'grau_instrucao_companheiro',
    'local_onde_reside', 
    'local_onde_trabalha',
]

# Loop through each column in the list
for col in features_nao_contínuas:
    missing = df[col].isna().sum()
    print(f"\nDistribution of '{col}' (valores ausentes: {missing})")

    # Contagem das ocorrências (ignorando os NaN)
    counts = df[col].value_counts(dropna=True).sort_index()

    # Criação de um DataFrame para exibição
    data = pd.DataFrame({
        'value': counts.index.astype(str),
        'occurrences': counts.values
    })

    # Display the chart
    plt.figure(figsize=(12, 5))
    sns.barplot(data=data, x='value', y='occurrences', hue='value', palette='viridis', legend=False)
    plt.title(f"Distribution of '{col}' (NaN: {missing})")
    plt.xlabel(col)
    plt.ylabel("Number of Occurrences")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
# Continuous variables to analyze
variaveis = ['idade', 'meses_na_residencia']

for var in variaveis:
    # Basic statistics
    data = df[var]
    min_val = data.min()
    max_val = data.max()
    mean_val = data.mean()
    median_val = data.median()
    nan_count = data.isna().sum()

    # Display the chart
    plt.figure(figsize=(8, 5))
    sns.histplot(data=data, kde=True, bins=30, color='skyblue')
    plt.title(f"Distribution of '{var}'")
    plt.xlabel(var)
    plt.ylabel("Frequency")

    # Exibição das linhas de estatísticas
    plt.axvline(min_val, color='red', linestyle='--', label=f'Mín: {min_val:.2f}')
    plt.axvline(max_val, color='green', linestyle='--', label=f'Máx: {max_val:.2f}')
    plt.axvline(mean_val, color='orange', linestyle='--', label=f'Média: {mean_val:.2f}')
    plt.axvline(median_val, color='purple', linestyle='--', label=f'Mediana: {median_val:.2f}')
    
    # Display the number of NaN values
    plt.legend(title=f"NaN: {nan_count}", loc='upper right')
    plt.grid(True)
    plt.tight_layout()
    plt.show()


By observing these distributions, we can compile a list of features for which the data appears highly imbalanced or incorrect. For example, qtde_contas_bancarias and qtde_contas_bancarias_especiais have the same distribution, so one of them should be removed:

- tipo_endereco — entirely filled with "1"
- grau_instrucao — entirely filled with zeros
- nacionalidade — entirely filled with "1"
- tipo_residencia — entirely filled with "1"
- possui_telefone_celular — entirely filled with "N"
- possui_cartao_diners — entirely filled with zeros
- possui_cartao_amex — entirely filled with zeros
- possui_outros_cartoes — entirely filled with zeros
- qtde_contas_bancarias
- qtde_contas_bancarias_especiais — identical content to 'qtde_contas_bancarias'
- estado_onde_trabalha
- codigo_area_telefone_trabalho
- meses_no_trabalho — entirely filled with zeros
- profissao — entirely filled with '9'
- profissao_companheiro — entirely filled with zeros
- grau_instrucao_companheiro — entirely filled with zeros

We will compute the feature importance using LightGBM and compare it with this list to determine which features are truly useless and can be removed.

## Feature Importance with LGBMClassifier

This code prepares the data for training a LightGBM model. It encodes categorical columns with LabelEncoder to convert categories into numbers. Then it separates the data into explanatory variables (X) and target variable (y).

The LGBMClassifier model is then trained on this data. After training, each variable's importance is extracted via feature_importances_. This way, we identify the most important features for model training, as well as those that can be removed.

In [ ]:
# Preparation
df_model = train_df.copy()

# Simple encoding of categorical columns
cat_cols = df_model.select_dtypes(include='object').columns
for col in cat_cols:
    df_model[col] = LabelEncoder().fit_transform(df_model[col].astype(str))

# Separação X / y
X = df_model.drop(columns=['id_solicitante', 'inadimplente'])
y = df_model['inadimplente']

# Initial model
model = LGBMClassifier(
    force_row_wise=True,
    random_state=42,
    n_estimators=100,
    learning_rate=0.1
)
model.fit(X, y)

# Importância
importances = pd.Series(model.feature_importances_, index=X.columns)
importances_sorted = importances.sort_values(ascending=False)

# Seleção das 10 variáveis mais importantes e 10 menos importantes
top_10 = importances_sorted.head(10)
bottom_15 = importances_sorted.tail(15)

# Exibição das 10 variáveis mais importantes
plt.figure(figsize=(10, 5))
top_10.plot(kind='barh', color='green')
plt.title("Top 10 Most Important Features (LightGBM)")
plt.gca().invert_yaxis()
plt.xlim(0, 420)
plt.grid(True)
plt.tight_layout()
plt.show()

# Exibição das 15 variáveis menos importantes
plt.figure(figsize=(10, 5))
bottom_15.plot(kind='barh', color='red')
plt.title("Top 15 Least Important Features (LightGBM)")
plt.gca().invert_yaxis()
plt.xlim(0,100)
plt.grid(True)
plt.tight_layout()
plt.show()


The feature importance results confirm our exploratory analysis of the distributions. The next step is to find a balance between removing the least useful features without eliminating too many.

In [ ]:
# Selection by threshold
limite_importancia = 5
features_a_manter = importances_sorted[importances_sorted >= limite_importancia].index.tolist()

# Colunas a manter + o alvo + eventualmente id_solicitante
colunas_a_manter_treino = features_a_manter + ['inadimplente', 'id_solicitante']
colunas_a_manter_teste = features_a_manter + ['id_solicitante']

# Remove unwanted columns
train_df_reduzido = train_df[colunas_a_manter_treino].copy()
test_df_reduzido = test_df[colunas_a_manter_teste].copy()

num_variaveis_removidas = len(importances_sorted) - len(features_a_manter)
print(f"Número de variáveis mantidas: {len(features_a_manter)}")
print(f"Número de variáveis removidas: {num_variaveis_removidas}")

After several attempts, varying the number of removed features, the best balance is to eliminate the 10 least useful features (which are also those with extraction errors). They are:
- 'meses_no_trabalho'
- 'nacionalidade'
- 'tipo_endereco'
- 'possui_cartao_diners'
- 'possui_telefone_celular'
- 'grau_instrucao'
- 'possui_cartao_amex'
- 'qtde_contas_bancarias_especiais'
- 'possui_outros_cartoes'
- 'local_onde_trabalha'

## Data Preprocessing, Missing Value Treatment (Imputers), and Scaling

The preprocess function prepares DataFrames for training and prediction:

- It separates columns into categorical and numerical to apply appropriate imputations.

- On the first call (fit_imputers=True), it creates and fits the imputers, replacing missing values.

- Then, it encodes categorical columns using OneHot encoding, transforming each category into a binary column.

- It returns the transformed DataFrame along with the imputers, for reuse on the test DataFrame — ensuring that the same transformations applied to the training set are applied exactly to the test set.

In [ ]:
# --- Preprocessing ---
def preprocess(df, imputers=None, fit_imputers=False):
    df = df.copy()

    # Identificar as colunas categóricas e numéricas
    categorical_cols = df.select_dtypes(include='object').columns.tolist()
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

    # Imputation of missing values
    if fit_imputers:
        # Create imputers for fitting
        cat_imputer = SimpleImputer(strategy='most_frequent')  # Replace missing categorical values with the most frequent
        num_imputer = SimpleImputer(strategy='median')         # Replace missing numerical values with the median

        # Aplicar os imputers e guardar para reutilização futura
        df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])
        df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

        imputers = {'cat_imputer': cat_imputer, 'num_imputer': num_imputer}
    else:
        # Reutilizar os imputers já treinados passados como parâmetro
        df[categorical_cols] = imputers['cat_imputer'].transform(df[categorical_cols])
        df[numeric_cols] = imputers['num_imputer'].transform(df[numeric_cols])

    # Codificação OneHot das colunas categóricas (removendo a primeira categoria para evitar multicolinearidade)
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

    return df, imputers


- We start by separating the explanatory variables X and the target y.

- Preprocessing is applied to the training set with imputer creation (missing value imputation + encoding).

- For the test set, we apply the same imputers to maintain transformation consistency.

- Then we harmonize columns between train and test: we add missing columns in test (filled with zeros) and remove extra columns.

- Finally, we ensure that the column order is identical in both DataFrames.

In [ ]:
# Separate features and target
X = train_df_reduzido.drop(columns=['id_solicitante', 'inadimplente'])
y = train_df_reduzido['inadimplente']

# Training preprocessing (imputation + encoding)
X_processado, imputers = preprocess(X, fit_imputers=True)

# Test preprocessing (with the same imputers)
test_processado, _ = preprocess(test_df_reduzido.drop(columns=['id_solicitante']), imputers=imputers, fit_imputers=False)

# --- Train/test harmonization ---
# Add missing columns in test_processed with zeros
colunas_faltando = set(X_processado.columns) - set(test_processado.columns)
for col in colunas_faltando:
    test_processado[col] = 0

# Remove extra columns that exist in test_processed but not in X_processed
colunas_extras = set(test_processado.columns) - set(X_processado.columns)
test_processado.drop(columns=colunas_extras, inplace=True)

# Reordenar as colunas do test_processado para corresponder à ordem de X_processado
test_processado = test_processado[X_processado.columns]

Finally, we apply standard normalization (mean 0, standard deviation 1) on the training data and transform the test data with the same scaler to ensure consistent scaling.

Scaling is not strictly necessary for the models used (Random Forest, XGBoost, and LightGBM), but after testing with and without scaling, results are better with the scaler.

In [ ]:
# --- Scaling ---
scaler = StandardScaler()
# Fit the scaler on training data e transformar
X_proc = pd.DataFrame(scaler.fit_transform(X_processado), columns=X_processado.columns)
# Transform test data with the same scaler
test_proc = pd.DataFrame(scaler.transform(test_processado), columns=test_processado.columns)

This line splits the preprocessed data (X_processed and y) into two sets:

- A training set (X_train, y_train) containing 80% of the data,

- A validation set (X_val, y_val) with the remaining 20%.

Validation is used to evaluate the model's performance on data unseen during training.

The random_state=42 parameter fixes the random seed to ensure reproducibility.

In [ ]:
# --- Train / validation split ---
X_train, X_val, y_train, y_val = train_test_split(X_proc, y, test_size=0.2, random_state=42)

## Model Training and Accuracy / Cross-Validation Computation

This block trains three different models:

Random Forest,  
XGBoost,  
LightGBM  

Each model is trained on the training data and then tested on the validation set.

Accuracy (accuracy_score) is computed to compare their performance on this validation.

In [ ]:
# --- Modeling (Random Forest + XGBoost) ---
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_val)
print("Acurácia Random Forest:", accuracy_score(y_val, y_pred_rf))

xgb = XGBClassifier(eval_metric='logloss')
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_val)
print("Acurácia XGBoost:", accuracy_score(y_val, y_pred_xgb))

# LightGBM
lgbm = LGBMClassifier(random_state=42, verbose=-1)
lgbm.fit(X_train, y_train)
y_pred_lgbm = lgbm.predict(X_val)
print("Acurácia LightGBM:", accuracy_score(y_val, y_pred_lgbm))

In [ ]:
# --- Cross-validation ---
# Random Forest
scores_rf = cross_val_score(rf, X_proc, y, cv=5, scoring='accuracy')
print("Acurácia da validação cruzada (Random Forest):", scores_rf.mean())

# XGBoost
scores_xgb = cross_val_score(xgb, X_proc, y, cv=5, scoring='accuracy')
print("Acurácia da validação cruzada (XGBoost):", scores_xgb.mean())

# LightGBM
scores_lgbm = cross_val_score(lgbm, X_proc, y, cv=5, scoring='accuracy')
print("Acurácia da validação cruzada (LightGBM):", scores_lgbm.mean())


In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score

# --- Cross-validation ---
scores_rf = cross_val_score(rf, X_proc, y, cv=10, scoring='accuracy')
scores_xgb = cross_val_score(xgb, X_proc, y, cv=10, scoring='accuracy')
scores_lgbm = cross_val_score(lgbm, X_proc, y, cv=10, scoring='accuracy')

# Data preparation for the boxplot
accuracies = [scores_rf, scores_xgb, scores_lgbm]
classif_models = ['Random Forest', 'XGBoost', 'LightGBM']
n_runs = len(scores_rf)  # number of folds

# Style et affichage
plt.style.use('_mpl-gallery')
fig, axs = plt.subplots(figsize=(10, 5))
axs.boxplot(accuracies, notch=True, labels=classif_models, patch_artist=True)
axs.set_title(f'Accuracies for {n_runs} folds', fontsize=14)
axs.set_ylabel('Accuracy')
plt.grid(True)
plt.tight_layout()
plt.show()

The LightGBM model is clearly more efficient than the other two; this is the one we will optimize for prediction.

## Optimized Hyperparameter Search

- This code defines a hyperparameter grid to test (such as number of leaves, max depth, learning rate, etc.).

- Then, it uses RandomizedSearchCV to randomly explore 50 combinations from this grid, with 3-fold cross-validation, optimizing for accuracy.

- After the search, it displays the best parameters found, as well as the best mean accuracy obtained during cross-validation.

This step improves model performance by finding the best hyperparameter configuration without exhaustively testing all combinations.

In [ ]:
# Base model definition
lgbm = LGBMClassifier(random_state=42, verbose=-1)

# Grade de parâmetros para testar (exemplo)
param_dist = {
    'num_leaves': [31, 50, 70, 90, 120],
    'max_depth': [7, 10, 15, 20, 25],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'n_estimators': [200, 400, 600, 800],
    'min_child_samples': [10, 20, 30, 40],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.1, 0.5, 1],
    'reg_lambda': [0, 0.1, 0.5, 1]
}

# Configuração do RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_dist,
    n_iter=50,
    scoring='accuracy',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Iniciar a busca
random_search.fit(X_train, y_train)

# Melhores parâmetros e score
print("Melhores parâmetros:", random_search.best_params_)
print("Melhor acurácia na validação cruzada:", random_search.best_score_)


## Prediction on the Test Set and Submission File Preparation

In [ ]:
# --- Training a model with optimized hyperparameters ---
best_params = {
    'subsample': 0.8,
    'reg_lambda': 1,
    'reg_alpha': 0.5,
    'num_leaves': 50,
    'n_estimators': 800,
    'min_child_samples': 20,
    'max_depth': 10,
    'learning_rate': 0.01,
    'colsample_bytree': 0.7,
    'random_state': 42,
    'verbose': -1
}

final_lgbm = LGBMClassifier(**best_params)
final_lgbm.fit(X_proc, y)

In [ ]:
# --- Prediction on test set and submission preparation ---
y_test_pred = final_lgbm.predict(test_proc)

submission = pd.DataFrame({
    'id_solicitante': test_df['id_solicitante'],
    'inadimplente': y_test_pred
})

submission.to_csv('submission.csv', index=False)
print("Arquivo de submissão salvo: submission.csv")

## Conclusion

The final accuracy score obtained via Kaggle submission was: **0.5996**

During this project, a wide variety of approaches were tested to improve the score:

- Feature engineering (creation of new variables),

- Ensemble methods such as Stacking and Voting with multiple models (but since LightGBM was clearly superior, including Random Forest and XGBoost in the vote reduced the score),

- Various imputation attempts with most frequent values, means, etc. (median remained the best),

- Hyperparameter optimization attempts with Optuna instead of RandomizedSearchCV, among other strategies.

Ultimately, this notebook produced the best result: without adding new features, using classical imputers and a basic hyperparameter search approach.

## Key Results

| Metric | Value |
|--------|-------|
| **Best model** | LightGBM |
| **Kaggle accuracy** | 0.5996 |
| **Cross-validation accuracy** | ~0.60 |
| **Features removed** | 10 out of 42 (least important + extraction errors) |
| **Key finding** | Classical imputation (median) + LightGBM outperformed complex ensemble approaches |

# Complete Source Code in a Single Cell

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from lightgbm import LGBMClassifier
from sklearn.impute import SimpleImputer

train_df = pd.read_csv('data/conjunto_de_treinamento.csv')
test_df = pd.read_csv('data/conjunto_de_teste.csv')
sample_submission = pd.read_csv('data/exemplo_arquivo_respostas.csv')

df_model = train_df.copy()
cat_cols = df_model.select_dtypes(include='object').columns
for col in cat_cols:
    df_model[col] = LabelEncoder().fit_transform(df_model[col].astype(str))

X = df_model.drop(columns=['id_solicitante', 'inadimplente'])
y = df_model['inadimplente']

model = LGBMClassifier(force_row_wise=True, random_state=42, n_estimators=100, learning_rate=0.1)
model.fit(X, y)

importances = pd.Series(model.feature_importances_, index=X.columns)
importances_sorted = importances.sort_values(ascending=False)

limite_importancia = 5
features_a_manter = importances_sorted[importances_sorted >= limite_importancia].index.tolist()

colunas_a_manter_treino = features_a_manter + ['inadimplente', 'id_solicitante']
colunas_a_manter_teste = features_a_manter + ['id_solicitante']

train_df_reduzido = train_df[colunas_a_manter_treino].copy()
test_df_reduzido = test_df[colunas_a_manter_teste].copy()

def preprocess(df, imputers=None, fit_imputers=False):
    df = df.copy()
    categorical_cols = df.select_dtypes(include='object').columns.tolist()
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

    if fit_imputers:
        cat_imputer = SimpleImputer(strategy='most_frequent')
        num_imputer = SimpleImputer(strategy='median')
        df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])
        df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])
        imputers = {'cat_imputer': cat_imputer, 'num_imputer': num_imputer}
    else:
        df[categorical_cols] = imputers['cat_imputer'].transform(df[categorical_cols])
        df[numeric_cols] = imputers['num_imputer'].transform(df[numeric_cols])

    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    return df, imputers

X = train_df_reduzido.drop(columns=['id_solicitante', 'inadimplente'])
y = train_df_reduzido['inadimplente']

X_processado, imputers = preprocess(X, fit_imputers=True)
test_processado, _ = preprocess(test_df_reduzido.drop(columns=['id_solicitante']), imputers=imputers, fit_imputers=False)

colunas_faltando = set(X_processado.columns) - set(test_processado.columns)
for col in colunas_faltando:
    test_processado[col] = 0
colunas_extras = set(test_processado.columns) - set(X_processado.columns)
test_processado.drop(columns=colunas_extras, inplace=True)
test_processado = test_processado[X_processado.columns]

scaler = StandardScaler()
X_proc = pd.DataFrame(scaler.fit_transform(X_processado), columns=X_processado.columns)
test_proc = pd.DataFrame(scaler.transform(test_processado), columns=test_processado.columns)

X_train, X_val, y_train, y_val = train_test_split(X_proc, y, test_size=0.2, random_state=42)

best_params = {
    'subsample': 0.8,
    'reg_lambda': 1,
    'reg_alpha': 0.5,
    'num_leaves': 50,
    'n_estimators': 800,
    'min_child_samples': 20,
    'max_depth': 10,
    'learning_rate': 0.01,
    'colsample_bytree': 0.7,
    'random_state': 42,
    'verbose': -1
}

final_lgbm = LGBMClassifier(**best_params)
final_lgbm.fit(X_proc, y)

y_test_pred = final_lgbm.predict(test_proc)

submission = pd.DataFrame({
    'id_solicitante': test_df['id_solicitante'],
    'inadimplente': y_test_pred
})

submission.to_csv('submission.csv', index=False)
print("Arquivo de submissão salvo: submission.csv")